# 用 Sciverse 做论文图表提取与分析 Demo

> 先检索论文，再从全文 Markdown 中定位图表路径并下载，结合多模态模型分析

**场景**: 研究人员需要从论文中提取特定图表并进行分析。流程是先通过语义检索找到相关论文，再从全文中定位图表路径，下载后用多模态模型提取信息。

**使用接口**: agentic-search, content, resource

**预估调用量**: ~8–20 次 API 调用

---


## Step 1: 环境准备

安装依赖并配置环境变量


In [ ]:
!pip install httpx anthropic
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值
import os
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # 替换为你的真实值


## Step 2: 检索论文并提取图表路径

先通过语义检索找到相关论文，再从全文中定位图表


In [ ]:
import os
import re
import asyncio
import httpx
from pathlib import Path

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def search_papers(query: str, top_k: int = 10):
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{BASE}/agentic-search", headers=HEADERS,
            json={"query": query, "top_k": top_k}
        )
        resp.raise_for_status()
        return resp.json()["hits"]

async def get_figures_from_doc(doc_id: str):
    """读取全文并提取图表路径"""
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(
            f"{BASE}/content", headers=HEADERS,
            params={"doc_id": doc_id, "offset": 0, "limit": 4000}
        )
        resp.raise_for_status()
        text = resp.json()["text"]  # 注意：字段是 text
        figure_paths = re.findall(r'!\\[.*?\\]\\((.*?)\\)', text)
        return figure_paths

async def main():
    hits = await search_papers("AlphaFold2 protein structure prediction accuracy")
    print(f"Found {len(hits)} relevant papers")
    # 对 top 3 论文提取图表
    all_figures = []
    for hit in hits[:3]:
        paths = await get_figures_from_doc(hit["doc_id"])
        print(f"  {hit['title'][:50]}: {len(paths)} figures")
        all_figures.extend([(hit["doc_id"], p) for p in paths])
    return all_figures

all_figures = asyncio.run(main())


## Step 3: 下载图表并用多模态模型分析

调用 resource 下载图片，传给多模态 LLM 分析


In [ ]:
import base64
from anthropic import Anthropic

async def download_figure(file_name: str, save_dir: str = "./figures"):
    Path(save_dir).mkdir(exist_ok=True)
    async with httpx.AsyncClient(timeout=60) as client:
        resp = await client.get(
            f"{BASE}/resource", headers=HEADERS,
            params={"file_name": file_name}  # 参数是 file_name
        )
        resp.raise_for_status()
        local = f"{save_dir}/{file_name.split('/')[-1]}"
        Path(local).write_bytes(resp.content)
        return local

def analyze_figure(image_path: str, question: str) -> str:
    """用多模态 LLM 分析图表"""
    client = Anthropic()
    with open(image_path, "rb") as f:
        img_data = base64.b64encode(f.read()).decode()

    msg = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=2048,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {
                    "type": "base64", "media_type": "image/png", "data": img_data
                }},
                {"type": "text", "text": question}
            ]
        }]
    )
    return msg.content[0].text

async def main():
    if all_figures:
        doc_id, path = all_figures[0]
        try:
            local = await download_figure(path)
            analysis = analyze_figure(local, "请描述这张图表的主要发现，提取关键数值。")
            print(f"\
Figure from {doc_id}:\
{analysis}")
        except httpx.HTTPStatusError as e:
            print(f"Download failed: {e.response.status_code}")

asyncio.run(main())


## 注意事项

- 这不是按视觉内容直接检索图表的功能，而是：先找论文 → 提取图表路径 → 下载 → 分析
- resource 接口参数是 file_name，传入 content 中提取的相对路径
- 部分论文可能没有可下载的图表资源（resource 返回 404）
- 多模态分析质量取决于图表清晰度和 LLM 能力
- 建议对图表分析结果做结构化提取（JSON schema）便于下游使用


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
